In [1]:
import caf.base as cb
import caf.tem as ct
import pandas as pd
import os
from pathlib import Path

# HB Production
## Preprocessing
### Create equivalent population DVector

In [2]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\lu_pop_2023.hdf"
if not os.path.exists(dvec_path):
    pop = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\landuse\lu_pop_2023.csv")
    pop = pop.set_index(["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"])
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"],
                                            naming_order=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("lsoa_2021")
    pop_dvec = cb.DVector(segmentation=segmentation, import_data=pop, zoning_system=zoning_system)
    pop_dvec.save(dvec_path)

### Create equivalent trip rates DVector

In [3]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\hb_trip_rates_production.hdf"
if not os.path.exists(dvec_path):
    tr = pd.read_csv(r"I:\NTS\outputs\productions\hb\trip_rates\hb_trip_rates_production.csv")
    tr = tr.rename(columns={"gender": "gender_3", "ns": "ns_sec", "purpose": "p"}).pivot(index=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"], columns="tfn_at", values="beta")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"],
                                                        naming_order=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    tr_dvec = cb.DVector(segmentation=segmentation, import_data=tr, zoning_system=zoning_system)
    tr_dvec.save(dvec_path)

### Create equivalent 2023 adjustment DVector

In [4]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\trip_rate_adjustments_production_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"I:\NTS\outputs\others\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create equivalent mts DVector

In [5]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\hb_mode_time_split_production_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"I:\NTS\outputs\productions\hb\mode_time_splits\hb_mode_time_split_production_fr_reg.csv")
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp", "hh_type"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp", "hh_type"],
                                                        naming_order=["p", "m", "tp", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

## TEM Setup

In [6]:
tem = ct.TEM(
    model_years=[2023],
    scenario="Core",
    output_zoning="normits",
    iteration_name="HBProd_01",
    export_home=r"T:\ThomasPrince\TEM I-Drive Comparison\Outputs - caf.tem",
    return_segmentation=["hh_type", "p", "m", "tp"]
)

## HB Production Model Setup and Run

In [7]:
input_dir = Path(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction")

HBProd = tem.HBProductionModel(
    population_paths={2023: input_dir / "lu_pop_2023.hdf"},
    trip_rates_path=input_dir / "hb_trip_rates_production.hdf",
    mode_time_splits_path=input_dir / "hb_mode_time_split_production_fr_reg.hdf",
    adjustment_path=input_dir / "trip_rate_adjustments_production_hb_fr.hdf",
    population_translation_path=r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\normits_lsoa_2021_pop.csv"
)

In [ ]:
#HBProd.run()

In [8]:
check = cb.DVector.load(HBProd.model.export_paths.pure_demand[2023])
check.aggregate(["p"]).data

normits_id,1001001,1001002,1001003,1001004,1001005,1001006,1001007,1001008,1001009,1001010,...,11357010,11358001,11358002,11358003,11358004,11358005,11358006,11358007,11358008,11358009
p,,,,,,,,,,,,,,,,,,,,,
1,17767.311173,11454.003375,10092.879569,8507.347114,9650.002728,10468.565819,10832.666399,10540.686855,21401.792761,17145.158193,...,39251.416794,6049.374697,201623.491312,171373.016608,78577.528696,124196.664546,187486.235632,178507.634490,95704.487677,145548.668551
2,2013.170122,1092.190241,972.267130,798.568497,927.934183,1095.043672,1287.106066,977.778471,2644.010471,1850.857204,...,4013.127439,846.187386,20529.140838,17463.498367,8003.813922,12651.406756,19086.116823,18211.080147,9740.177919,14844.718919
3,11249.375749,9629.294894,8469.263260,6654.506597,7663.942013,7699.855037,6732.681385,9487.632847,12592.195916,10673.471481,...,21556.205035,4012.639347,96541.387495,90526.492800,40245.871451,60033.483857,74559.866041,85057.778802,51937.570815,72659.248538
4,20018.179529,16409.541120,14974.651592,12033.725270,13850.436001,14000.420199,13957.937891,15782.001166,27003.097498,21728.571332,...,50142.069143,6868.728729,214612.413545,195308.782067,85580.091988,132280.338315,168170.578406,189890.003005,105713.729538,157451.617095
5,8199.879878,6398.801505,5952.118688,4643.985739,5522.757540,5630.115709,5932.932142,6040.849941,11720.585240,8868.288892,...,21371.651025,3009.294243,88230.126877,81711.092934,35522.233126,54476.432713,66483.383493,78292.515827,44080.057135,65279.653161
6,13505.559921,8947.694374,8188.853387,6426.635867,7652.350967,8285.088484,9226.764112,8223.761054,18610.356903,13320.901708,...,31000.856209,4848.298492,138286.858015,124961.713729,55437.397445,85440.440550,112537.397985,122623.553114,68794.911867,101871.666688
7,9546.328532,7910.131347,6960.718628,5692.065979,6488.685535,6634.106118,6328.032368,7674.000016,12037.569294,9871.222599,...,21858.268046,3145.787058,95135.302913,87495.305606,38579.182124,58787.169697,74168.798827,84356.340216,48226.836493,70496.636257
8,6379.099740,4391.077743,3914.946947,3187.540528,3731.468042,4035.279874,4424.733366,4066.418717,8846.009281,6421.625327,...,9398.772672,2340.720728,41168.715514,37521.942648,16573.320589,25428.369309,32703.023664,36612.293984,20570.503224,30413.506392


In [9]:
pop_2023 = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\reports\pop_2023_normits.csv")
pop_2023 = pop_2023.groupby(["normits_v3.3_id"])[["1","2","3","4","5","6","7","8"]].sum().T
pop_2023.index = pop_2023.index.astype(int)
pop_2023

normits_v3.3_id,1001001,1001002,1001003,1001004,1001005,1001006,1001007,1001008,1001009,1001010,...,11357010,11358001,11358002,11358003,11358004,11358005,11358006,11358007,11358008,11358009
1,17767.311173,11454.003375,10092.879569,8507.347114,9650.002728,10468.565819,10832.666399,10540.686855,21401.792761,17145.158193,...,39251.416794,6049.374697,201623.491312,171373.016608,78577.528696,124196.664546,187486.235632,178507.634490,95704.487677,145548.668551
2,2013.170122,1092.190241,972.267130,798.568497,927.934183,1095.043672,1287.106066,977.778471,2644.010471,1850.857204,...,4013.127439,846.187386,20529.140838,17463.498367,8003.813922,12651.406756,19086.116823,18211.080147,9740.177919,14844.718919
3,11249.375749,9629.294894,8469.263260,6654.506597,7663.942013,7699.855037,6732.681385,9487.632847,12592.195916,10673.471481,...,21556.205035,4012.639347,96541.387495,90526.492800,40245.871451,60033.483857,74559.866041,85057.778802,51937.570815,72659.248538
4,20018.179529,16409.541120,14974.651592,12033.725270,13850.436001,14000.420199,13957.937891,15782.001166,27003.097498,21728.571332,...,50142.069143,6868.728729,214612.413545,195308.782067,85580.091988,132280.338315,168170.578406,189890.003005,105713.729538,157451.617095
5,8199.879878,6398.801505,5952.118688,4643.985739,5522.757540,5630.115709,5932.932142,6040.849941,11720.585240,8868.288892,...,21371.651025,3009.294243,88230.126877,81711.092934,35522.233126,54476.432713,66483.383493,78292.515827,44080.057135,65279.653161
6,13505.559921,8947.694374,8188.853387,6426.635867,7652.350967,8285.088484,9226.764112,8223.761054,18610.356903,13320.901708,...,31000.856209,4848.298492,138286.858015,124961.713729,55437.397445,85440.440550,112537.397985,122623.553114,68794.911867,101871.666688
7,9546.328532,7910.131347,6960.718628,5692.065979,6488.685535,6634.106118,6328.032368,7674.000016,12037.569294,9871.222599,...,21858.268046,3145.787058,95135.302913,87495.305606,38579.182124,58787.169697,74168.798827,84356.340216,48226.836493,70496.636257
8,6379.099740,4391.077743,3914.946947,3187.540528,3731.468042,4035.279874,4424.733366,4066.418717,8846.009281,6421.625327,...,9398.772672,2340.720728,41168.715514,37521.942648,16573.320589,25428.369309,32703.023664,36612.293984,20570.503224,30413.506392


In [10]:
test = (check.aggregate(["p"]).data - pop_2023).stack()
test = test.reset_index()
test = test.groupby("normits_id")[0].sum()
test.loc[abs(test)>1] # Where the difference in total trips, by normits id, is > 1

normits_id
5248009    2715.089252
Name: 0, dtype: float64

There's one normits zone to look into...\
5248009\
NB. this normits zone has two tfn_at's - this is where the error will stem from...

In [11]:
check = cb.DVector.load(HBProd.model.export_paths.tem_segmented[2023])
check.aggregate(["p"]).data

normits_id,1001001,1001002,1001003,1001004,1001005,1001006,1001007,1001008,1001009,1001010,...,11357010,11358001,11358002,11358003,11358004,11358005,11358006,11358007,11358008,11358009
p,,,,,,,,,,,,,,,,,,,,,
1,13399.534867,8638.241099,7611.725285,6415.967682,7277.721810,7895.055776,8169.648728,7949.447143,16140.544032,12930.327093,...,29602.156613,4562.243909,152057.954951,129244.019482,59260.645875,93665.131478,141396.091231,134624.718911,72177.247707,109768.126431
2,1866.190829,1012.450657,901.282998,740.265907,860.186749,1015.095761,1193.135896,906.391962,2450.974233,1715.728196,...,3720.133506,784.408194,19030.331292,16188.507939,7419.464445,11727.741739,17692.660842,16881.509612,9029.058454,13760.922642
3,13014.103291,11139.874885,9797.865173,7698.421507,8866.210464,8907.757285,7788.859838,10975.989835,14567.576191,12347.854978,...,24937.799675,4642.115625,111686.160800,104727.689272,46559.377144,69451.139095,86256.323883,98401.079651,60085.192849,84057.550097
4,17552.161975,14388.067768,13129.940706,10551.303869,12144.215999,12275.723808,12238.474849,13837.833772,23676.615570,19051.852490,...,43965.122707,6022.577579,188174.545987,171248.907673,75037.574432,115984.868695,147453.829521,166497.662054,92690.971281,138055.325283
5,7569.984168,5907.260448,5494.890767,4287.245550,5098.512144,5197.623308,5477.178087,5576.805891,10820.237128,8187.047555,...,19729.930474,2778.127255,81452.493631,75434.236721,32793.497752,50291.679788,61376.284521,72278.266760,40693.929615,60265.021955
6,15079.560845,9990.500396,9143.220542,7175.625976,8544.191628,9250.671327,10302.094223,9182.196511,20779.294670,14873.381704,...,34613.840537,5413.341804,154403.453227,139525.334496,61898.330227,95398.067868,125653.031065,136914.673740,76812.591661,113744.265713
7,6922.151316,5735.726141,5047.296188,4127.381734,4705.019628,4810.465745,4588.528190,5564.504629,8728.578306,7157.735697,...,15849.678587,2281.045949,68983.689388,63443.840509,27974.203424,42627.244883,53780.638984,61167.741029,34969.827255,51117.912168
8,10308.685606,7096.023227,6326.591350,5151.095686,6030.087702,6521.050500,7150.411033,6571.371166,14295.234791,10377.407357,...,15188.505668,3782.626869,66529.034239,60635.814743,26782.643062,41092.485672,52848.396019,59165.813885,33242.128062,49148.514420


In [12]:
notem_2023 = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\reports\NoTEM_2023_GB.csv")
notem_2023 = notem_2023.loc[notem_2023["direction"]=="hb_fr"].pivot(["purpose"])

,direction,purpose,mode,period,hh_type,soc,ns,prod,attr
0,hb_fr,Commuting,Bus,1,ca,1,1,455023.869493,455023.869493
1,hb_fr,Commuting,Bus,1,ca,1,2,106747.884332,106747.884332
2,hb_fr,Commuting,Bus,1,ca,1,3,35490.306543,35490.306543
3,hb_fr,Commuting,Bus,1,ca,1,4,3812.645994,3812.645994
4,hb_fr,Commuting,Bus,1,ca,1,5,1642.174468,1642.174468
...,...,...,...,...,...,...,...,...,...
40315,nhb,Visit friends,Walk,6,nca,4,1,7397.204990,7397.204990
40316,nhb,Visit friends,Walk,6,nca,4,2,9786.482323,9786.482323
40317,nhb,Visit friends,Walk,6,nca,4,3,19623.173301,19623.173301
40318,nhb,Visit friends,Walk,6,nca,4,4,14653.207313,14653.207313


# HB Attraction
## Preprocessing
### Create equivalent trip rates DVectors

In [12]:
tr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_attraction.csv")
tr = tr.loc[tr["dir"]=="hb"]

# Segmentations
soc = cb.Segmentation(cb.SegmentationInput(enum_segments=["soc"], naming_order=["soc"]))
sic = cb.Segmentation(cb.SegmentationInput(enum_segments=["sic_2_digit"], naming_order=["sic_2_digit"]))
total = cb.Segmentation(cb.SegmentationInput(enum_segments=["total"], naming_order=["total"]))
tfn_at = cb.ZoningSystem.get_zoning("tfn_at")

#p1
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf"):
    tr1 = tr.loc[tr["p"]==1]
    tr1 = tr1.pivot(index = "soc", columns="tfn_at", values="alpha")
    tr1.index = tr1.index.astype(int)
    cb.DVector(segmentation=soc, import_data=tr1, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf")

#p2
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf"):
    tr2 = tr.loc[tr["p"]==2]
    tr2 = tr2.pivot(index = "soc", columns="tfn_at", values="alpha")
    tr2.index = tr2.index.astype(int)
    cb.DVector(segmentation=soc, import_data=tr2, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf")

#p3
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf"):
    tr3 = tr.loc[tr["p"]==3]
    tr3["sic_2_digit"] = 85
    tr3 = tr3.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr3, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf")

#p4
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf"):
    tr4 = tr.loc[tr["p"]==4]
    tr4["sic_2_digit"] = tr4.loc[:, "e_code"].apply(lambda x: [46, 47])
    tr4 = tr4.explode("sic_2_digit")
    tr4 = tr4.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr4, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf")

#p5
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf"):
    tr5 = tr.loc[tr["p"]==5]
    tr5_1 = tr5.loc[tr5["e_code"]=="e08"].copy()
    tr5_1["sic_2_digit"] = 86
    tr5_2 = tr5.loc[tr5["e_code"]=="e09"].copy()
    tr5_2["sic_2_digit"] = tr5_2.loc[:, "e_code"].apply(lambda x: [64, 65, 66, 68, 69, 75, 77, 79, 80, 95, 96])
    tr5_2 = tr5_2.explode("sic_2_digit")
    tr5_3 = tr5.loc[tr5["e_code"]=="e11"].copy()
    tr5_3["sic_2_digit"] = 56
    tr5 = pd.concat([tr5_1,tr5_2,tr5_3])
    tr5 = tr5.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr5, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf")

#p6
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf"):
    tr6 = tr.loc[tr["p"]==6]
    tr6["sic_2_digit"] = tr6["e_code"].apply(lambda x: [90, 91, 92, 93, 94])
    tr6 = tr6.explode("sic_2_digit")
    tr6 = tr6.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr6, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf")

#p7
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf"):
    tr7 = tr.loc[tr["p"]==7]
    tr7["total"] = 1
    tr7 = tr7.pivot(index = "total", columns="tfn_at", values="alpha")
    tr7.index = tr7.index.astype(int)
    cb.DVector(segmentation=total, import_data=tr7, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf")

#p8
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf"):
    tr8 = tr.loc[tr["p"]==8]
    tr8["sic_2_digit"] = tr8.loc[:, "e_code"].apply(lambda x: [2, 3, 55])
    tr8 = tr8.explode("sic_2_digit")
    tr8 = tr8.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr8, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf")


### Create Equivalent trip rates adjustment

In [13]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments_attractions_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create Equivalent MTS DVec

In [14]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.csv")
    mts = mts.loc[mts["uni"]==0]
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp"],
                                                        naming_order=["p", "m", "tp"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

### Create Equivalent MTS Adjustment

In [15]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p", "period": "tp", "mode": "m"}).loc[(adj["pa"]=="a") & (adj["direction"]=="hb_fr")].pivot(index=["p", "tp", "m"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "tp", "m"],
                                                        naming_order=["p", "tp", "m"])) # TODO delete and rewrite adj with "a"
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

## HB Attraction Model setup

In [16]:
HBAttr = tem.HBAttractionModel(
    trip_rates_paths={
        1: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf",
        2: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf",
        3: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf",
        4: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf",
        5: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf",
        6: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf",
        7: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf",
        8: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf"
    },
    balance_production=True,
    emp_landuse_paths = {2023: r"F:\Deliverables\Land-Use\241213_Employment\02_Final Outputs\Output E6.hdf"},
    hh_landuse_dirs = {2023: r"F:\Deliverables\Land-Use\241220_Populationv2\02_Final Outputs"},
    hh_landuse_prefix = "Output P13.3",
    mode_time_splits_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.hdf",
    trip_rate_adjustment_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments_attractions_hb_fr.hdf",
    hh_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_pop.csv",
    emp_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_emp.csv",
    mode_time_splits_adjustment_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.hdf"
)

In [ ]:
HBAttr.run()

In [28]:
check = cb.DVector.load(HBAttr.model.export_paths.mts_demand_adj[2023])
mdl_attr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\mdlnotem_outputs\emp_2023_normits.csv")
mdl_attr = mdl_attr.set_index("normits_v3.3_id")[["1","2","3","4","5","6","7","8"]].T
mdl_attr.index = mdl_attr.index.astype(int)

In [ ]:
check.aggregate(["p"]).data#.sum()

In [ ]:
mdl_attr#.sum().sum()

In [ ]:
(check.data / mdl_attr).stack().describe()

In [24]:
test = (check.data / mdl_attr).stack()
test.loc[abs(test)>1.00001]

Again, it is only this one zone where any difference in trips occur

In [ ]:
End of testing.